In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Turbine anomaly detection

In [0]:
SILVER_TABLE = "interviews_dev.silver.turbine_clean"
ANOMALY_TABLE = "interviews_dev.gold.turbine_anomalies"

In [0]:

silver_df = spark.table(SILVER_TABLE)

# Add the daily period used for the 24-hour calculation.
readings_df = silver_df.withColumn(
    "reading_date",
    F.to_date("timestamp")
)

# One statistical population per turbine per day.
daily_turbine_window = Window.partitionBy(
    "turbine_id",
    "reading_date"
)

# Add mean and standard deviation without collapsing individual readings.
anomaly_scored_df = (
    readings_df
    .withColumn(
        "mean_power_output_mw",
        F.avg("power_output").over(daily_turbine_window)
    )
    .withColumn(
        "stddev_power_output_mw",
        F.stddev_samp("power_output").over(daily_turbine_window)
    )
    .withColumn(
        "lower_power_threshold_mw",
        F.col("mean_power_output_mw")
        - (F.lit(2.0) * F.col("stddev_power_output_mw"))
    )
    .withColumn(
        "upper_power_threshold_mw",
        F.col("mean_power_output_mw")
        + (F.lit(2.0) * F.col("stddev_power_output_mw"))
    )
    .withColumn(
        "z_score",
        F.when(
            F.col("stddev_power_output_mw") > 0,
            (
                F.col("power_output")
                - F.col("mean_power_output_mw")
            ) / F.col("stddev_power_output_mw")
        )
    )
    .withColumn(
        "is_anomaly",
        F.abs(F.col("z_score")) > 2
    )
    .withColumn(
        "anomaly_type",
        F.when(F.col("z_score") > 2, F.lit("HIGH_POWER_OUTPUT"))
        .when(F.col("z_score") < -2, F.lit("LOW_POWER_OUTPUT"))
    )
)

In [0]:
anomalies_df = (
    anomaly_scored_df
    .filter(F.col("is_anomaly"))
    .select(
        "timestamp",
        "turbine_id",
        "reading_date",
        "power_output",
        "wind_speed",
        "wind_direction",
        "mean_power_output_mw",
        "stddev_power_output_mw",
        "lower_power_threshold_mw",
        "upper_power_threshold_mw",
        "z_score",
        "anomaly_type"
    )
)

(
    anomalies_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .saveAsTable(ANOMALY_TABLE)
)

In [0]:
display(
    anomaly_scored_df
    .groupBy("turbine_id")
    .agg(
        F.sum(F.when(F.col("is_anomaly"), 1).otherwise(0)).alias("anomaly_count")
    )
    .filter(F.col("anomaly_count") == 0)
    .withColumnRenamed("turbine_id","turbines_with_no_anomalies")
    .select("turbines_with_no_anomalies")
)

In [0]:
# Anomalies only
anomalies_df = anomaly_scored_df.filter(F.col("is_anomaly") == True)

# Per-turbine anomaly summary
turbine_anomaly_summary = (
    anomalies_df
    .groupBy("turbine_id")
    .agg(
        F.count("*").alias("total_anomalies"),
        F.sum(
            F.when(F.col("anomaly_type") == "LOW_POWER_OUTPUT", 1).otherwise(0)
        ).alias("low_power_anomalies"),
        F.sum(
            F.when(F.col("anomaly_type") == "HIGH_POWER_OUTPUT", 1).otherwise(0)
        ).alias("high_power_anomalies")
    )
    .orderBy(F.col("total_anomalies").desc())
)

display(turbine_anomaly_summary)

Databricks visualization. Run in Databricks to view.

# Summary statistics

In [0]:
# Add the daily period used for the 24-hour calculation.
readings_df = silver_df.withColumn(
    "reading_date",
    F.to_date("timestamp")
)

# Per-turbine, per-day summary statistics
turbine_daily_summary = (
    readings_df
    .groupBy("turbine_id", "reading_date")
    .agg(
        F.min("power_output").alias("min_power_output_mw"),
        F.max("power_output").alias("max_power_output_mw"),
        F.avg("power_output").alias("avg_power_output_mw"),
        F.count("*").alias("reading_count")
    )
    .orderBy("turbine_id", "reading_date")
)

display(turbine_daily_summary)

In [0]:
last_day = turbine_daily_summary.agg(F.max("reading_date")).first()["max(reading_date)"]

plot_df = (
    turbine_daily_summary
    .filter(F.col("reading_date") == last_day)
    .orderBy("turbine_id")
)

display(plot_df)

Databricks visualization. Run in Databricks to view.

In [0]:
DAILY_SUMMARY_TABLE = "interviews_dev.gold.turbine_daily_summary"

(
    turbine_daily_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DAILY_SUMMARY_TABLE)
)